# Full Parameter Fine Tuning (FPFT) GPT-2 on MedText

This notebook documents how GPT-2 is fine-tuned to answer questions from the MedText dataset. The workflow covers data preparation, tokenization, baseline evaluation, hyperparameter search, loss visualization, and a qualitative comparison of generated answers.

**Validation loss** measures how well the model predicts the tokens in the reference answers. The final comparison of generated answers is qualitative; neither result alone establishes clinical accuracy.

## 1. Dataset preparation

All settings (model, dataset, splits, training, both searches and generation) live in `configs/gpt2_medtext.yaml`. `load_config()` reads it into typed dataclasses, and every function below receives that `config`.

The `prepare_dataset()` function in `src/dataset_preparation.py` processes the `BI55/MedText` dataset:

1. It resolves the current dataset revision and loads the `Prompt` and `Completion` fields.
2. It strips surrounding whitespace, removes empty examples and duplicate prompts, and renames the fields to `prompt` and `completion`.
3. It splits the examples into **training (80%)**, **validation (10%)**, and **test (10%)** sets using seed `42`.
4. It saves the three splits to `data/processed/medtext_v1` so they can be reused in later runs.

This step cleans and splits the data. **Data augmentation is not implemented** in the current version.

In [1]:
from src.config import load_config
from src.dataset_preparation import prepare_dataset

config = load_config("configs/gpt2_medtext.yaml")
prepare_dataset(config)

/home/hpc3/Documents/Projects/medllm-finetuning/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


data/processed/medtext_v1 already exists


### Load the dataset splits

The notebook loads the processed dataset. In this run, it contains **1,129 training examples**, **141 validation examples**, and **142 test examples**. The training set is used to update model weights; the validation set is used to compare trials and select a checkpoint; the test set is reserved for an independent final evaluation.

If the processed directory already exists, `prepare_dataset()` does not recreate it.

In [2]:
from src.dataset_preparation import load_medtext

train_data, val_data, test_data = load_medtext(config.data.processed_dir)

## 2. Prompt-completion format

`to_sft_format()` (in `src/tokenizer.py`) rewrites each `prompt` as `Question:` followed by the question and `Answer:`, and keeps the `completion`. This is the *prompt-completion* format of TRL's `SFTTrainer`, which:

- tokenizes the examples with GPT-2's `AutoTokenizer` and appends the end-of-text token to the answer;
- computes **the loss only on the answer** (`completion_only_loss=True`), although the model sees the question as context;
- pads each batch to its longest example.

The same format is used for training, validation, and test, so every loss in this notebook is computed the same way.


In [3]:
from src.formatting import to_sft_format

train_sft = to_sft_format(train_data)
val_sft = to_sft_format(val_data)
test_sft = to_sft_format(test_data)


Map: 100%|██████████| 142/142 [00:00<00:00, 44166.94 examples/s]


## 3. Baseline model and validation loss

The pretrained `openai-community/gpt2` model (about **124 million parameters**) is loaded onto the available device. Before fine-tuning, `evaluate()` (in `src/evaluation.py`) computes its validation loss with `SFTTrainer.evaluate()`, using a fixed evaluation batch size of 8. The grid search uses the same batch size for its validation losses, so they are directly comparable to this baseline.

In this run, the **baseline validation loss was 2.8858**.


In [4]:
import math

from transformers import AutoTokenizer
from src.modeling import load_model
from src.evaluation import evaluate
from src.tracking import configure_tracking, log_evaluation

configure_tracking(config)

# [Step 1] Load the pretrained weights from Hugging Face
model, device = load_model(config.model.model_id)
tokenizer = AutoTokenizer.from_pretrained(config.model.model_id)

# [Step 2] Evaluate the model before fine-tuning and record it in MLflow
baseline_loss = evaluate(model, tokenizer, val_sft, config)
log_evaluation(
    config.model.model_id,
    "validation",
    {"loss": baseline_loss, "perplexity": math.exp(baseline_loss)},
    config,
)


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 19303.95it/s]


Device: cuda
Parameters: 124,439,808


Truncating train dataset: 100%|██████████| 141/141 [00:00<00:00, 30440.44 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 141/141 [00:00<00:00, 85486.68 examples/s]
/home/hpc3/Documents/Projects/medllm-finetuning/.venv/lib/python3.11/site-packages/triton/knobs.py:504: UserWarning: TRITON_DEFAULT_FP_FUSION is unset: defaulting to FP fusion ENABLED for backward compatibility in Triton 3.9. The default will change to DISABLED in Triton 3.10; set TRITON_DEFAULT_FP_FUSION=1 to keep fusion enabled or =0 to adopt the new default early and silence this warning.
  warn_once(FP_FUSION_DEFAULT_WARNING)


Training Loss,Validation Loss,Step,Entropy,Mean Token Accuracy,Num Tokens
No log,2.885836,0,3.076833,0.431362,0.000000


## 4. Fine-tuning and hyperparameter search

`run_grid_search()` evaluates **3 batch sizes** (`2`, `8`, and `16`) × **3 learning rates** (`1e-5`, `2e-5`, and `5e-5`): nine trials. Each trial trains a fresh pretrained GPT-2 with TRL's `SFTTrainer`, using `AdamW` with `weight_decay=0.01`, a constant learning rate, and a gradient norm clipped to `1.0`.

Each trial runs for at most **20 epochs** with **early stopping**: training stops once validation loss has not improved for **2 consecutive epochs**, so the number of epochs is chosen by the data rather than fixed in advance. Training and validation losses are logged after each epoch. At the end of a trial, the trainer reloads the epoch with the lowest validation loss; if it beats every earlier trial, it is saved (with the tokenizer) to `checkpoints/gpt2-medtext-best`. Trial results are written to `experiments/hpo_results.json`.

The table below comes from the earlier run with the hand-written training loop; re-running this cell with `SFTTrainer` will produce new values.

| Trial | Batch size | Learning rate | Best validation loss | Epoch |
| ---: | ---: | ---: | ---: | ---: |
| 1 | 2 | `1e-5` | 2.1202 | 5 |
| 2 | 2 | `2e-5` | 2.0859 | 5 |
| **3** | **2** | **`5e-5`** | **2.0832** | **3** |
| 4 | 8 | `1e-5` | 2.1695 | 5 |
| 5 | 8 | `2e-5` | 2.1191 | 5 |
| 6 | 8 | `5e-5` | 2.0881 | 4 |

In this run, the best checkpoint came from **trial 3, epoch 3**. The difference from trial 2 is small (2.0832 versus 2.0859), so this single search does not establish that `5e-5` will consistently perform better.

**MLflow tracking.** Each search is one parent run (`grid-search` or `optuna-search`, tag `stage=finetune`) holding the configuration and both datasets, with one nested run per trial (hyperparameters, train/validation loss per epoch, CPU/GPU usage). Every run is tagged with the git commit. Only the best trial stores its model, which is registered as a new version of `gpt2-medtext-sft` with the alias `grid-best` or `optuna-best`, so later stages can load it with `load_model("models:/gpt2-medtext-sft@grid-best")`. Evaluations are separate runs with `stage=evaluate`. Open the UI with `uv run mlflow ui --backend-store-uri sqlite:///mlflow.db`.


In [5]:
from src.train import run_grid_search

best, results = run_grid_search(train_sft, val_sft, config)



Trial 1/9
Parameters: {'batch_size': 2, 'learning_rate': 1e-05}


Truncating train dataset: 100%|██████████| 1129/1129 [00:00<00:00, 39736.25 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 1129/1129 [00:00<00:00, 126899.16 examples/s]


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,2.572128,2.271291,2.367009,0.500112,162502.000000
2,2.303002,2.206467,2.224672,0.514830,325004.000000
3,2.166046,2.166992,2.089323,0.519916,487506.000000
4,2.059784,2.146899,1.989690,0.524662,650008.000000
5,1.966250,2.130733,1.930860,0.527340,812510.000000
6,1.883846,2.123998,1.880912,0.532810,975012.000000
7,1.810620,2.109605,1.795179,0.535487,1137514.000000
8,1.736597,2.111042,1.739390,0.536977,1300016.000000
9,1.672083,2.114691,1.682640,0.538110,1462518.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.52it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.85it/s]



Trial 2/9
Parameters: {'batch_size': 2, 'learning_rate': 2e-05}


Loading weights: 100%|██████████| 148/148 [00:00<00:00, 28602.36it/s]


... (27 outputs left out)

Loading weights: 100%|██████████| 148/148 [00:00<00:00, 21997.06it/s]


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,2.531888,2.225600,2.340815,0.510080,162502.000000
2,2.189474,2.153671,2.097539,0.520842,325004.000000
3,2.011731,2.115090,1.964448,0.527543,487506.000000
4,1.871346,2.106435,1.827269,0.533023,650008.000000
5,1.745998,2.092189,1.776306,0.537270,812510.000000
6,1.630924,2.088461,1.699643,0.541119,975012.000000
7,1.523545,2.104057,1.584347,0.540660,1137514.000000
8,1.422888,2.121037,1.477461,0.544431,1300016.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.08it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.56it/s]



Best hyperparameters:
{'trial': 6, 'mlflow_run_id': '1b5c3f87a8894feaadb06787ad76f8ae', 'batch_size': 8, 'learning_rate': 5e-05, 'best_val_loss': 2.0792741775512695, 'best_epoch': 4, 'train_losses': [2.4904327392578125, 2.1168937145824165, 1.9040649843887545, 1.7388121913856185, 1.5803968402701365, 1.4475872415891835], 'val_losses': [2.1916003227233887, 2.12457537651062, 2.0913877487182617, 2.0792741775512695, 2.098446846008301, 2.129131555557251]}


## 5. Loss visualization

The plot compares the **training** and **validation** curves for all six trials. Its third panel compares the best validation loss for each hyperparameter combination. The horizontal line in the validation panel marks the original GPT-2 validation loss (`baseline_loss`). The figure is also saved to `experiments/grid_search_losses.png`.

In trial 3, validation loss falls to **2.0832 at epoch 3** and then rises in epochs 4 and 5, while training loss keeps falling. This suggests overfitting after epoch 3. The checkpoint saved at the best epoch is therefore more relevant than the model at the end of that trial.

In [5]:
import importlib
import src.plot as plot_module

importlib.reload(plot_module)

plot_module.plot_grid_search(
    results,
    baseline_val_loss=baseline_loss,
)

NameError: name 'results' is not defined

### Alternative: Bayesian search with Optuna

`run_optuna_search()` reads its search space, number of trials and pruning settings from the `optuna` section of the config. It explores a wider space than the grid: learning rate (log-uniform between `1e-5` and `1e-4`), batch size (`2`, `4`, `8`, `16`), weight decay (`0`, `0.01`, `0.1`), learning-rate schedule (`constant` or `cosine`), and warmup (`0` or `10%` of the steps).

- **TPE sampler (seed `42`)**: after a few random trials, Optuna proposes combinations near the ones with the lowest validation loss so far, instead of testing every combination.
- **Median pruning**: after the first five trials, a trial whose validation loss at an epoch (from epoch 2 on) is worse than the median of earlier trials at that epoch is stopped early.

Each trial uses the same early stopping as the grid search (at most 20 epochs, patience 2), and its objective is its best validation loss. With early stopping, a `cosine` schedule is defined over the 20-epoch cap, so a trial that stops early ends before the learning rate has fully decayed. The best completed model is saved to `checkpoints/gpt2-medtext-optuna-best` and every trial (including pruned ones, with `status: "pruned"`) to `experiments/optuna_results.json`, so the grid's checkpoint and results are not overwritten.


In [6]:
from src.train import run_optuna_search

optuna_best, optuna_results, study = run_optuna_search(train_sft, val_sft, config)

study.trials_dataframe(attrs=("number", "value", "params", "state")).sort_values("value")


[I 2026-11-02 15:21:51,064] A new study created in memory with name: no-name-30c9ed18-2460-4d6e-ad82-a98bbdbc09d8



Trial 1/20
Parameters: {'learning_rate': 2.368863950364079e-05, 'batch_size': 2, 'weight_decay': 0.1, 'lr_scheduler_type': 'cosine', 'warmup_ratio': 0.1}


Truncating train dataset: 100%|██████████| 1129/1129 [00:00<00:00, 37966.78 examples/s]
Dropping fully masked examples from train dataset: 100%|██████████| 1129/1129 [00:00<00:00, 124024.23 examples/s]


Epoch,Training Loss,Validation Loss,Entropy,Mean Token Accuracy,Num Tokens
1,2.719101,2.327740,2.435233,0.492295,162502.000000
2,2.360951,2.205135,2.215488,0.512442,325004.000000
3,2.135884,2.140032,2.041311,0.526746,487506.000000
4,1.936970,2.112263,1.865016,0.532830,650008.000000
5,1.772911,2.099993,1.754073,0.535219,812510.000000
6,1.629365,2.103352,1.627918,0.542437,975012.000000
7,1.502986,2.114830,1.541001,0.544360,1137514.000000


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.50it/s]
[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.44it/s]
2026/11/02 15:23:55 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/hpc3/Documents/Projects/medllm-finetuning
2026/11/02 15:23:55 WARNING mlflow.transformers: The model card could not be retrieved from the hub due to Repo id must be in the form 'repo_name' or 'namespace/repo_name': 'checkpoints/optuna_trials/trial_1/mlflow_model'. Use `repo_type` argument if needed.
2026/11/02 15:23:55 WARNING mlflow.transformers: Unable to find license information for this model. Please verify permissible usage for the model you are storing prior to use.
2026/11/02 15:23:55 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Writing 


Trial 2/20
Parameters: {'learning_rate': 6.798962421591133e-05, 'batch_size': 16, 'weight_decay': 0.0, 'lr_scheduler_type': 'constant', 'warmup_ratio': 0.1}


... (73 outputs left out)

Epoch,Training Loss,Validation Loss


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  3.52it/s]
2026/11/02 15:39:11 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /home/hpc3/Documents/Projects/medllm-finetuning
2026/11/02 15:39:11 WARNING mlflow.transformers: The model card could not be retrieved from the hub due to Repo id must be in the form 'repo_name' or 'namespace/repo_name': 'checkpoints/optuna_trials/trial_20/mlflow_model'. Use `repo_type` argument if needed.
2026/11/02 15:39:11 WARNING mlflow.transformers: Unable to find license information for this model. Please verify permissible usage for the model you are storing prior to use.
2026/11/02 15:39:11 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
[I 2026-11-02 15:39:12,166] Trial 19 pruned. Pruned at epoch 1



Best hyperparameters:
{'trial': 15, 'status': 'complete', 'mlflow_run_id': '7c9023138cb64fb696490bf86a89f454', 'mlflow_model_id': 'm-3283d944868a4cfd9fba153ca9ef4fda', 'learning_rate': 5.1765173605918435e-05, 'batch_size': 8, 'weight_decay': 0.01, 'lr_scheduler_type': 'cosine', 'warmup_ratio': 0.0, 'best_val_loss': 2.079223155975342, 'best_epoch': 4, 'train_losses': [2.488034745337258, 2.1091495567644145, 1.8937850737235915, 1.7275516348825375, 1.571106816681338, 1.4451474471830985], 'val_losses': [2.1908600330352783, 2.1243062019348145, 2.088282823562622, 2.079223155975342, 2.099807024002075, 2.1261281967163086]}


,number,value,params_batch_size,params_learning_rate,params_lr_scheduler_type,params_warmup_ratio,params_weight_decay,state
14,14,2.079223,8,0.000052,cosine,0.0,0.01,COMPLETE
11,11,2.080708,2,0.000032,cosine,0.0,0.01,COMPLETE
12,12,2.081806,8,0.000050,cosine,0.0,0.01,COMPLETE
8,8,2.082279,2,0.000033,constant,0.1,0.00,COMPLETE
2,2,2.085684,2,0.000029,cosine,0.0,0.01,COMPLETE
17,17,2.086497,4,0.000037,constant,0.0,0.01,COMPLETE
16,16,2.087681,2,0.000056,cosine,0.0,0.00,COMPLETE
1,1,2.094097,16,0.000068,constant,0.1,0.00,COMPLETE
10,10,2.094189,2,0.000023,constant,0.1,0.00,COMPLETE
4,4,2.094766,4,0.000035,cosine,0.1,0.10,COMPLETE


## 6. Qualitative comparison of generated answers

The next cell loads the original GPT-2 model and the checkpoint with the best validation loss. For the **first ten questions in the validation split**, it displays the question, the reference answer, and both models' answers. Both models receive the same `Question:`/`Answer:` prompt format and generate up to `config.generation.max_new_tokens` (256) new tokens with deterministic decoding (`do_sample=False`).

Reviewing these outputs can reveal repetition, incoherence, or disagreement with the reference. In this run, for example, the fine-tuned model identifies the topic of the first case, but its answer to the second case differs from the reference. **These ten examples do not measure clinical accuracy.** They also come from the validation split, which was already used to select the checkpoint.

In [ ]:
from transformers import AutoModelForCausalLM
from src.inference import generate_response

base_model = AutoModelForCausalLM.from_pretrained(config.model.model_id).to(device)
finetuned_model = AutoModelForCausalLM.from_pretrained(
    config.grid_search.best_checkpoint_dir
).to(device)

for i in range(10):
    question = val_data[i]["prompt"]
    reference = val_data[i]["completion"]

    base_response = generate_response(
        base_model, tokenizer, question, device, config.generation.max_new_tokens
    )
    ft_response = generate_response(
        finetuned_model, tokenizer, question, device, config.generation.max_new_tokens
    )

    print(f"\nQUESTION:\n{question}")
    print(f"\nREFERENCE:\n{reference}")
    print(f"\nBASE MODEL:\n{base_response}")
    print(f"\nFINE-TUNED:\n{ft_response}")
    print("-" * 80)
